## `etcd` — the cluster's source of truth

**`etcd`** is a distributed key-value store. Every object you've created — pod, deployment, configmap, secret, namespace, node — is a row in `etcd`. **Lose `etcd`, lose the cluster's memory.**

Properties that matter:

- **Strongly consistent** — built on the Raft consensus algorithm; reads see the latest committed write.
- **HA with an odd number of members** — production runs 3 or 5; tolerates (N−1)/2 failures.
- **Ports `2379` (client) and `2380` (peer)** — both TLS.
- **Authenticates clients with certificates** — the API server uses `apiserver-etcd-client.crt`.

In a kubeadm cluster, `etcd` runs as a **static pod** (`/etc/kubernetes/manifests/etcd.yaml`), started before anything else. The alternative — **external etcd** on dedicated machines — is more moving parts but better blast-radius isolation; big managed offerings use it.

What's inside: every object, serialised, under keys like `/registry/pods/default/web-...`, `/registry/secrets/team-a/db-credentials`. You *can* poke with `etcdctl`, but in practice you go through the API server (`kubectl get`) — direct reads are for forensics and backups.

The one thing to internalise now: **`etcd` backup is the cluster's life insurance.** Lose it and there are no objects, no PV bindings, nothing. Backup is a `snapshot save`; restore is a `snapshot restore` plus repointing `etcd` at the new data dir — a skill the CKA tests reliably (we do the commands later this module). On our map this is the **etcd** component in the control plane — the one box whose loss is unrecoverable without a backup.